**MODULE 1 — Data Model (Customer)**

Goal

Define a clean, typed representation of a customer and attach business logic (tiering).

**Piece 1 — The Customer dataclass**

Before writing it, answer these questions in plain English:

What does a Customer have? — id, name, spent, status

What can a Customer do? — tell you its tier

That is it. Nothing else. Write only that:

In [3]:
from dataclasses import dataclass

@dataclass
class Customer:
    id     : int
    name   : str
    spent  : float
    status : str

    def get_tier(self):
        if self.spent > 50000:
            return "Platinum"
        elif self.spent > 20000:
            return "Gold"
        elif self.spent > 5000:
            return "Silver"
        else:
            return "Bronze"
c = Customer(301, "Priya Mehta", 45000, "active")
print(c)
print(c.get_tier())

Customer(id=301, name='Priya Mehta', spent=45000, status='active')
Gold


**Piece 2 — BasePipeline**

Before writing it, answer in plain English:

What does every pipeline have? — a name, a count of processed records, a count of skipped records

What can every pipeline do? — log a message, show a summary

In [5]:
class BasePipeline:

  def __init__(self,name):
    self.name = name
    self.processed = 0
    self.skipped = 0

  def log(self, message):
    print(f"[{self.name}] {message}")

  def summary(self):
    print(f"\n[{self.name}] Summary")
    print(f"  Processed : {self.processed}")
    print(f"  Skipped   : {self.skipped}")


# To Test
'''bp = BasePipeline("TestPipeline:")
bp.log("Hello from base")
bp.processed = 3
bp.skipped   = 1
bp.summary()'''

'bp = BasePipeline("TestPipeline:")\nbp.log("Hello from base")\nbp.processed = 3\nbp.skipped   = 1\nbp.summary()'

**Piece 3 — The generator**

This is where most people get confused. Let me show you the thinking.

A generator processes one record at a time. For each record it asks three questions in order:
```
Question 1 — is the name empty?    → if yes, skip
Question 2 — can spent convert?    → if no, skip  
Question 3 — is status valid?      → if no, skip
If all three pass → yield a Customer object

In [7]:
import logging

logging.basicConfig(
    level  = logging.INFO,
    format = "%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger("CustomerPipeline")

def clean_customers(raw_records):

    valid_statuses = {"active", "inactive"}

    for raw in raw_records:

        # Question 1 — is name empty?
        name = raw.get("name", "").strip()
        if not name:
            logger.error(f"Skipped record {raw.get('id')} — name is empty")
            continue

        # Question 2 — can spent convert to float?
        try:
            spent = float(raw["spent"])
        except ValueError:
            logger.error(f"Skipped record {raw.get('id')} — spent '{raw['spent']}' is not a number")
            continue

        # Question 3 — is status valid?
        status = raw.get("status", "").strip().lower()
        if status not in valid_statuses:
            logger.error(f"Skipped record {raw.get('id')} — status '{status}' is invalid")
            continue

        # All three passed — yield a clean Customer object
        yield Customer(
            id     = int(raw["id"]),
            name   = name,
            spent  = spent,
            status = status
        )

In [8]:
raw_customers = [
    {"id": "301", "name": "  Priya Mehta  ", "spent": "45000", "status": "active"},
    {"id": "302", "name": "Arjun Patel",     "spent": "bad",   "status": "active"},
    {"id": "303", "name": "  Sneha Shah  ",  "spent": "12000", "status": "inactive"},
    {"id": "304", "name": "Rahul Verma",     "spent": "72000", "status": "Active"},
    {"id": "305", "name": "",                "spent": "9000",  "status": "active"},
    {"id": "306", "name": "Divya Nair",      "spent": "31000", "status": "invalid"},
]

for customer in clean_customers(raw_customers):
    print(customer)

ERROR:CustomerPipeline:Skipped record 302 — spent 'bad' is not a number
ERROR:CustomerPipeline:Skipped record 305 — name is empty
ERROR:CustomerPipeline:Skipped record 306 — status 'invalid' is invalid


Customer(id=301, name='Priya Mehta', spent=45000.0, status='active')
Customer(id=303, name='Sneha Shah', spent=12000.0, status='inactive')
Customer(id=304, name='Rahul Verma', spent=72000.0, status='active')


**Piece 4 — CustomerPipeline**

Now the pipeline. It inherits from `BasePipeline` so it gets `log()` and `summary()` for free. It only needs to define `run()`.

Before writing — what does `run()` do in plain English?
```
- log that it is starting
- loop through clean_customers() generator
- for each valid customer:
    - increment processed count
    - print the customer and their tier
- at the end call summary()

In [10]:
class CustomerPipeline(BasePipeline):

    def __init__(self):
        super().__init__("CustomerPipeline")

    def run(self, raw_customers):
        self.log("Starting")

        total_raw = len(raw_customers)

        for customer in clean_customers(raw_customers):
            self.processed += 1
            self.log(f"Valid: {customer.name} | Tier: {customer.get_tier()}")

        self.skipped = total_raw - self.processed

        self.summary()

pipeline = CustomerPipeline()
pipeline.run(raw_customers)

ERROR:CustomerPipeline:Skipped record 302 — spent 'bad' is not a number
ERROR:CustomerPipeline:Skipped record 305 — name is empty
ERROR:CustomerPipeline:Skipped record 306 — status 'invalid' is invalid


[CustomerPipeline] Starting
[CustomerPipeline] Valid: Priya Mehta | Tier: Gold
[CustomerPipeline] Valid: Sneha Shah | Tier: Silver
[CustomerPipeline] Valid: Rahul Verma | Tier: Platinum

[CustomerPipeline] Summary
  Processed : 3
  Skipped   : 3
